# TsTests — 时间序列统计检验工具包

本 Notebook 演示 `TsTests` 包的完整用法，涵盖以下检验类别：

1. **标准单位根检验** — ADF, Phillips-Perron, KPSS
2. **结构性突变单位根检验** — Perron (1989), Zivot-Andrews (1992)
3. **ARCH 效应检验** — Ljung-Box Q, Engle LM
4. **正态性检验与白噪音检验** — Jarque-Bera, Ljung-Box (raw)
5. **协整检验** — Johansen 迹检验与最大特征根检验
6. **格兰杰因果检验** — Toda-Yamamoto (1995)

所有检验类继承 `BaseTest` (ABC)，遵循 `fit()` / `summary()` / `result_` 统一契约。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Ts.TsTests import (
    ADFTest, PhillipsPerronTest, KPSSTest,
    PerronTest, ZivotAndrewsTest,
    LjungBoxTest, EngleLMTest, NormalityTest,
    JohansenTest, TodaYamamotoTest,
)
from Ts.TsSims import simulate_cointegrated

# 构造测试数据
np.random.seed(42)
random_walk = np.cumsum(np.random.randn(200))          # 随机游走（单位根）
ar1_stationary = np.zeros(200)
ar1_stationary[0] = np.random.randn()
for t in range(1, 200):
    ar1_stationary[t] = 0.5 * ar1_stationary[t-1] + np.random.randn()   # 平稳 AR(1)

# 协整数据 (k=2, r=1) — 供 Johansen 检验使用
r_coint = simulate_cointegrated(n=300, k=2, coint_rank=1, seed=42)
coint_data = r_coint.get_data().values

In [ ]:
# ARCH 数据 (用 GARCH with q=0 生成)
from Ts.TsSims import simulate_garch
arch_data = simulate_garch(n=500, p=1, q=0, omega=0.4, alpha=[0.5], seed=42, burn=200).data
noise = np.random.randn(500)  # 无 ARCH 效应的白噪声


---

## 1. 标准单位根检验

### 1.1 ADF 检验 — 随机游走 (无法拒绝 H0)

In [ ]:
test = ADFTest(random_walk, trend="c", lags=1)
print(test.summary())

# 可视化
test.result_.plot_test()
plt.show()


### 1.2 ADF 检验 — 平稳 AR(1) (拒绝 H0)

In [ ]:
test = ADFTest(ar1_stationary, trend="c", lags=1)
print(test.summary())

test.result_.plot_test()
plt.show()


### 1.3 ADF with trend="ct" — 趋势平稳 vs 差分平稳

In [ ]:
# 趋势平稳过程
t = np.arange(200)
ts_data = 5.0 + 0.3 * t + np.random.randn(200) * 2

test_ts = ADFTest(ts_data, trend="ct", lags=1)
print("趋势平稳 (TS) 过程:")
print(test_ts.summary())

# 差分平稳过程 (随机游走)
test_ds = ADFTest(random_walk, trend="ct", lags=1)
print("\n差分平稳 (DS) 过程:")
print(test_ds.summary())


### 1.4 ADF 自动滞后选择 (AIC / BIC)

In [ ]:
test_aic = ADFTest(random_walk, trend="c", max_lags=12, autolag="AIC")
test_bic = ADFTest(random_walk, trend="c", max_lags=12, autolag="BIC")

r_aic = test_aic.fit()
r_bic = test_bic.fit()

print(f"AIC: 选用滞后 {r_aic.lags}, IC = {r_aic.icbest:.2f}")
print(f"BIC: 选用滞后 {r_bic.lags}, IC = {r_bic.icbest:.2f}")


### 1.5 Phillips-Perron 检验

In [ ]:
print("=== PP on Random Walk ===")
pp_rw = PhillipsPerronTest(random_walk, trend="c")
print(pp_rw.summary())

print("\n=== PP on Stationary AR(1) ===")
pp_ar = PhillipsPerronTest(ar1_stationary, trend="c")
print(pp_ar.summary())


### 1.6 PP rho 检验

In [ ]:
pp_rho = PhillipsPerronTest(random_walk, trend="c", test_type="rho")
print(pp_rho.summary())


### 1.7 KPSS 检验 — H0: 平稳 (假设反转!)

In [ ]:
print("=== KPSS on Random Walk ===")
print("(H0: stationary → should REJECT → unit root)")
kpss_rw = KPSSTest(random_walk, trend="c")
print(kpss_rw.summary())

print("\n=== KPSS on Stationary AR(1) ===")
print("(H0: stationary → should NOT reject → stationary)")
kpss_ar = KPSSTest(ar1_stationary, trend="c")
print(kpss_ar.summary())


### 1.8 KPSS with trend="ct" — 趋势平稳检验

In [ ]:
t = np.arange(200)
ts_process = 10.0 + 0.5 * t + np.random.randn(200) * 3

kpss_ct = KPSSTest(ts_process, trend="ct")
print(kpss_ct.summary())

kpss_ct.result_.plot_test()
plt.show()


### 1.9 三种单位根检验对比

In [ ]:
datasets = {
    "Random Walk": random_walk,
    "Stationary AR(1)": ar1_stationary,
    "Trend-Stationary": ts_process,
}

print(f"{'Dataset':<20s} {'ADF p':<10s} {'PP p':<10s} {'KPSS p':<10s}")
print("-" * 50)
for name, data in datasets.items():
    adf_p = ADFTest(data, trend="c", lags=1).fit().pvalue
    pp_p  = PhillipsPerronTest(data, trend="c").fit().pvalue
    kpss_p = KPSSTest(data, trend="c").fit().pvalue
    print(f"{name:<20s} {adf_p:<10.4f} {pp_p:<10.4f} {kpss_p:<10.4f}")

print("\n注意: ADF/PP H0=单位根 (p小→平稳), KPSS H0=平稳 (p小→单位根)")


---

## 2. 结构性突变单位根检验

### 2.1 Perron (1989) 检验 — 已知断点

In [ ]:
# 构造带结构性突变的数据
np.random.seed(123)
n = 100
years = np.arange(1920, 1920 + n, dtype=float)
break_year = 1950.0

# 截距突变模型
y = np.zeros(n)
y[0] = 10.0
break_idx = np.argmin(np.abs(years - break_year))
for t in range(1, n):
    y[t] = 0.3 + y[t - 1] + np.random.randn() * 2
    if t >= break_idx:
        y[t] += 15  # 断点后截距突变

# Perron 检验 — intercept 模型
pt = PerronTest(y, break_year=break_year, time_index=years, model="intercept", max_lags=8)
print(pt.summary())


### 2.2 Perron 检验 — 三种模型对比

In [ ]:
for model in ["intercept", "slope", "both"]:
    pt = PerronTest(y, break_year=break_year, time_index=years, model=model)
    r = pt.fit()
    reject = "Reject H0" if r.statistic < r.cv_05 else "Cannot reject H0"
    print(f"Model: {model:<12s}  t(rho)={r.statistic:7.3f}  CV(5%)={r.cv_05:6.3f}  -> {reject}")


### 2.3 Zivot-Andrews (1992) 检验 — 未知断点

In [ ]:
# Zivot-Andrews 检验 — 内生选择断点
za = ZivotAndrewsTest(y, time_index=years, model="intercept", max_lags=8, trim=0.15)
print(za.summary())

# t-statistic 序列图
za.result_.plot_test()
plt.show()


### 2.4 Zivot-Andrews with AIC lag selection

In [ ]:
za_aic = ZivotAndrewsTest(
    y, time_index=years, model="both",
    max_lags=8, lag_method="aic", trim=0.15,
)
print(za_aic.summary())

# IC 图 (lag selection)
za_aic.result_.plot_test()
plt.show()


---

## 3. ARCH 效应检验

### 3.1 Ljung-Box Q 检验 — ARCH 效应 (应拒绝 H0)

In [ ]:
lb_arch = LjungBoxTest(arch_data, lags=10)
print("ARCH data (has volatility clustering):")
print(lb_arch.summary())

lb_noise = LjungBoxTest(noise, lags=10)
print("\nWhite noise (no ARCH effects):")
print(lb_noise.summary())


### 3.2 Engle LM 检验 — ARCH 效应

In [ ]:
lm_arch = EngleLMTest(arch_data, lags=10)
print("ARCH data (should detect ARCH effects):")
print(lm_arch.summary())

lm_noise = EngleLMTest(noise, lags=10)
print("\nWhite noise (should NOT detect ARCH effects):")
print(lm_noise.summary())


### 3.3 Engle LM — 不同滞后阶数的影响

In [ ]:
print(f"{'Lags':<8s} {'LM stat':<12s} {'p-value':<10s} {'ARCH?'}")
print("-" * 45)
for lag in [1, 5, 10, 20]:
    r = EngleLMTest(arch_data, lags=lag).fit()
    has_arch = "YES" if r.pvalue < 0.05 else "NO"
    print(f"{lag:<8d} {r.statistic:<12.4f} {r.pvalue:<10.4f} {has_arch}")


---

## 4. 正态性检验与白噪音检验

### 4.1 Jarque-Bera 正态性检验 — 正态数据 (不应拒绝 H0)

In [ ]:
jb_normal = NormalityTest(noise)
print("Normal data (white noise):")
print(jb_normal.summary())

# 正态分布的残差也应通过检验
jb_resid = NormalityTest(np.random.randn(300))
print("\nNormal residuals:")
print(jb_resid.summary())


### 4.2 非正态数据 — t 分布与 ARCH 数据 (应拒绝 H0)

In [ ]:
# t(3) 分布 — 厚尾，应拒绝正态性 H0
t_data = np.random.standard_t(df=3, size=500)
jb_t = NormalityTest(t_data)
print("t(3) data (heavy-tailed, should reject normality):")
print(jb_t.summary())

# ARCH 数据 — 尖峰厚尾，应拒绝正态性
jb_arch = NormalityTest(arch_data)
print("\nARCH data (leptokurtic, should reject normality):")
print(jb_arch.summary())


### 4.2b NormalityTest.plot_test() — 可视化

`NormalityTestResult` 提供 `plot_test()` 方法，绘制直方图与正态密度曲线叠加图。

In [ ]:
# NormalityTest 可视化
jb_plot = NormalityTest(t_data)
jb_plot.fit()
jb_plot.result_.plot_test()
plt.show()

### 4.3 白噪音检验 — Ljung-Box on raw residuals

`LjungBoxTest` 默认 `apply_squared=True` 检验 ARCH 效应（对平方残差）。
设置 `apply_squared=False` 直接检验原始残差是否存在自相关 — 即白噪音检验。

In [ ]:
# 白噪音 — 不应拒绝 H0 (无自相关)
wn_noise = LjungBoxTest(noise, lags=10, apply_squared=False)
print("White noise (no autocorrelation expected):")
print(wn_noise.summary())

# AR(1) — 应拒绝 H0 (存在自相关)
wn_ar = LjungBoxTest(ar1_stationary, lags=10, apply_squared=False)
print("\nAR(1) data (autocorrelation expected):")
print(wn_ar.summary())


### 4.4 正态性与白噪音对比表

In [ ]:
print(f"{'Data':<20s} {'JB stat':<10s} {'JB p':<10s} {'Q stat':<10s} {'Q p':<10s}")
print("-" * 60)
datasets_test = {
    "Normal Noise": noise,
    "t(3) Heavy-Tail": t_data,
    "ARCH Data": arch_data,
    "AR(1) Stationary": ar1_stationary,
}
for name, data in datasets_test.items():
    jb_r = NormalityTest(data).fit()
    wn_r = LjungBoxTest(data, lags=10, apply_squared=False).fit()
    print(f"{name:<20s} {jb_r.statistic:<10.2f} {jb_r.pvalue:<10.4f} {wn_r.statistic:<10.2f} {wn_r.pvalue:<10.4f}")


---

## 5. 协整检验与格兰杰因果检验

`TsTests` 提供两类多变量检验：

- **Johansen 协整检验** — 确定协整秩 `r`，包含迹检验和最大特征根检验
- **Toda-Yamamoto 格兰杰因果检验** — 无需单位根/协整预检验，适合 I(0)/I(1)/协整系统

### 5.1 Johansen 协整秩检验 — 迹检验 + 最大特征根

用 `simulate_cointegrated` 生成 k=2, r=1 的协整数据，验证 Johansen 检验恢复协整秩。

In [ ]:
# ============================================================
# Johansen 协整检验: k=2, r=1
# ============================================================
joh = JohansenTest(coint_data, lags=2, trend="constant")
print(joh.summary())

# 迹检验: H0: r <= r0
#   若 r=0 被拒绝 (stat > 5% crit)、r<=1 未被拒绝 → rank = 1
print(f"\n→ 协整秩 (trace) = {joh.result_.rank_trace}")
print(f"→ 协整秩 (max-eig) = {joh.result_.rank_maxeig}")

### 5.2 不同确定性趋势设定

Johansen 检验支持 5 种趋势设定，对应不同的数据生成假设：

In [ ]:
print(f"{'Trend':<12s} {'Trace Rank':<12s} {'Max-Eig Rank':<14s} {'结论'}")
print("-" * 55)

for trend in ["none", "rconstant", "constant", "rtrend", "trend"]:
    try:
        t = JohansenTest(coint_data, lags=2, trend=trend)
        r = t.fit()
        note = "✓ r=1" if r.rank_trace == 1 and r.rank_maxeig == 1 else f"r_t={r.rank_trace}, r_m={r.rank_maxeig}"
    except Exception as e:
        note = f"Error: {e}"
    print(f"{trend:<12s} {note}")

print("\n提示: 'constant' (默认) 最常用，假设数据有线性趋势、协整方程有截距。")

### 5.3 Toda-Yamamoto 格兰杰因果检验

无需单位根/协整预检验，通过 `d_max` 额外滞后保证 Wald 统计量服从卡方分布。
下面构造 X → Y 的单向因果关系（VAR(1)，d_max=0）。

In [ ]:
# ============================================================
# 构造单向因果关系: X → Y (X 导致 Y, Y 不导致 X)
# VAR(1): X_t = 0.5 X_{t-1} + e_t
#          Y_t = 0.7 Y_{t-1} + 0.4 X_{t-1} + u_t
# ============================================================
np.random.seed(42)
n = 200
X = np.zeros(n)
Y = np.zeros(n)
for t in range(1, n):
    X[t] = 0.5 * X[t - 1] + np.random.randn()
    Y[t] = 0.7 * Y[t - 1] + 0.4 * X[t - 1] + np.random.randn()
causal_data = np.column_stack([X, Y])

print("=== 单向因果关系: X → Y ===")
print("期望: X 导致 Y (应拒绝 H0), Y 不导致 X (不应拒绝 H0)\n")
ty = TodaYamamotoTest(causal_data, p=1, d_max=0, cols=["X", "Y"])
print(ty.summary())

# 对于 I(1) 变量，设置 d_max=1
print("\n=== I(1) 变量演示: d_max=1 ===")
# 用随机游走构造 I(1) 数据
rw_x = np.cumsum(np.random.randn(200))
rw_y = np.cumsum(np.random.randn(200))
# 人为令 X 包含 Y 的滞后信息
rw_y[1:] = 0.5 * rw_y[:-1] + 0.3 * rw_x[:-1] + np.random.randn(199)
ty_i1 = TodaYamamotoTest(
    np.column_stack([rw_x, rw_y]), p=1, d_max=1, cols=["X", "Y"]
)
print(ty_i1.summary())

---

## 小结

| 检验 | 类名 | H0 | 来源 |
|------|------|----|------|
| ADF | `ADFTest` | 单位根 (非平稳) | Dickey-Fuller (1979) |
| Phillips-Perron | `PhillipsPerronTest` | 单位根 (非平稳) | Phillips-Perron (1988) |
| KPSS | `KPSSTest` | **平稳** (反转) | KPSS (1992) |
| Perron | `PerronTest` | 含断点的单位根 | Perron (1989) |
| Zivot-Andrews | `ZivotAndrewsTest` | 含断点的单位根 | ZA (1992) |
| Ljung-Box | `LjungBoxTest` | 无自相关 | Ljung-Box (1978) |
| Engle LM | `EngleLMTest` | 无 ARCH 效应 | Engle (1982) |
| Jarque-Bera | `NormalityTest` | 正态分布 | Jarque-Bera (1987) |
| Johansen | `JohansenTest` | 协整秩 ≤ r | Johansen (1988) |
| Toda-Yamamoto | `TodaYamamotoTest` | 无格兰杰因果 | Toda-Yamamoto (1995) |

**关键要点**:
- `LjungBoxTest(data, apply_squared=True)` → ARCH 效应检验 (m² 检验)
- `LjungBoxTest(data, apply_squared=False)` → 白噪音检验
- `JohansenTest` 同时运行迹检验和最大特征根检验，自动序贯确定协整秩
- `TodaYamamotoTest` 通过 `d_max` 额外滞后避免单位根/协整预检验偏差
- 所有单位根检验 Result 提供 `plot_test()` — 临界值对比图
- `NormalityTestResult` 提供 `plot_test()` — 直方图 + 正态密度曲线
- 所有检验类继承 `BaseTest` → `fit()`, `summary()`, `result_`